# Unmodified Model

Runs the twitter-roberta-base-sentiment model without modifications, which produces positive/negative instead of liberal/neutral

In [1]:
!pip install pyprojroot

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from transformers import AutoTokenizer, AutoConfig
from transformers import AutoModelForSequenceClassification
from transformers import pipeline

import polars as pl
import numpy as np
import time

from pyprojroot import here
from scipy.special import softmax
from ast import literal_eval

In [4]:
# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split():
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Setup model and tokenizer
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Load train and test data
train = pl.read_parquet("drive/MyDrive/Data/train_reduced.parquet")
test = pl.read_parquet("drive/MyDrive/Data/test_reduced.parquet")

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [6]:
# Test inference logic

tst = train.head(10).row(2, named=True)

print(tst)

text = preprocess(tst["content"])
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
scores = softmax(output[0][0].detach().numpy())
{label: score for label, score in zip(config.id2label.values(), scores)}

{'index': 20484533, 'id': 'ch0ymih', 'subreddit': 'Cooking', 'username': 'artard', 'username_score': 0.0, 'content': '"I love Alton, but he\'s off point.....this mother fucker right here makes it rain pepper:\\n\\nhttp://www.unicornmills.com/Magnum-Pepper-Mill-Black/\\n\\nI could have coated those flank steaks in the time it took him to remember where he put the drill."', 'label': 'neutral'}


{'negative': np.float32(0.8874582),
 'neutral': np.float32(0.093327716),
 'positive': np.float32(0.01921405)}

In [7]:
def inference_single(content: str) -> dict[str, np.float32]:
    text = preprocess(literal_eval(content))
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    output = model(**encoded_input)
    scores = softmax(output[0][0].detach().numpy())
    return {label: float(score) for label, score in zip(config.id2label.values(), scores)}

In [8]:
inference_single(test.head(10).row(2, named=True)["content"])

{'negative': 0.005612809676676989,
 'neutral': 0.7630065679550171,
 'positive': 0.2313806414604187}

In [9]:
test = test.with_columns(
    pl.col("content")
    .map_elements(inference_single, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

train = train.with_columns(
    pl.col("content")
    .map_elements(inference_single, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

In [10]:
calc_prediction = pl.when(pl.col("negative") > pl.col("neutral")).then(
        pl.when(pl.col("negative") > pl.col("positive"))
        .then(pl.lit("negative"))
        .otherwise(pl.lit("positive"))
    ).otherwise(pl.when(pl.col("neutral") > pl.col("positive"))
        .then(pl.lit("neutral"))
        .otherwise(pl.lit("positive"))
    ).alias("prediction")

train = train.with_columns(calc_prediction)
test = test.with_columns(calc_prediction)

In [11]:
train.group_by(["label", "prediction"]).agg(pl.len().alias("count")).with_columns((pl.col("count") / pl.col("count").sum().over("label")).alias("pct_of_label"))

label,prediction,count,pct_of_label
str,str,u32,f64
"""neutral""","""negative""",5647,0.232014
"""neutral""","""positive""",5554,0.228193
"""neutral""","""neutral""",13138,0.539792
"""liberal""","""positive""",1817,0.175861
"""liberal""","""negative""",2826,0.273519
"""conversative""","""negative""",3690,0.26344
"""conversative""","""neutral""",7851,0.560505
"""liberal""","""neutral""",5689,0.550619
"""conversative""","""positive""",2466,0.176055


In [12]:
test.group_by(["label", "prediction"]).agg(pl.len().alias("count")).with_columns((pl.col("count") / pl.col("count").sum().over("label")).alias("pct_of_label"))

label,prediction,count,pct_of_label
str,str,u32,f64
"""neutral""","""negative""",1421,0.229083
"""conversative""","""positive""",622,0.174523
"""neutral""","""neutral""",3333,0.537321
"""conversative""","""neutral""",1996,0.560045
"""conversative""","""negative""",946,0.265432
"""liberal""","""neutral""",1450,0.549451
"""liberal""","""positive""",472,0.178856
"""liberal""","""negative""",717,0.271694
"""neutral""","""positive""",1449,0.233597


In [13]:
train.group_by(["label"]).agg(pl.col("negative").mean(), pl.col("neutral").mean(), pl.col("positive").mean())

label,negative,neutral,positive
str,f64,f64,f64
"""neutral""",0.24788,0.488447,0.263673
"""conversative""",0.276757,0.506261,0.216982
"""liberal""",0.284649,0.502311,0.21304


In [14]:
test.group_by(["label"]).agg(pl.col("negative").mean(), pl.col("neutral").mean(), pl.col("positive").mean())

label,negative,neutral,positive
str,f64,f64,f64
"""neutral""",0.246498,0.486562,0.266941
"""conversative""",0.275335,0.507387,0.217278
"""liberal""",0.284045,0.505066,0.210889


In [15]:
train.write_parquet("drive/MyDrive/Data/train_reduced_twitter-roberta-sentiment.parquet")
test.write_parquet("drive/MyDrive/Data/test_reduced_twitter-roberta-sentiment.parquet")